# 14 · 예측 모델 구현과 평가 — 자동차 연비 맞히기
교과서 **Ⅱ-05(100~107쪽)** 의 «데이터 분리 → 모델 학습 → 모델 평가»를 **교과서와 다른 사례**로 다시 해 봅니다.

맨 먼저 **파일 → 드라이브에 사본 저장**을 하세요(내 드라이브에 저장돼야 고친 내용이 남습니다).
`____` 가 보이면 거기가 문제입니다. 빈칸 **바로 위 💡 힌트**를 읽고 채운 뒤 셀을 실행하세요(Shift + Enter).
교과서는 `files.upload()` 로 파일을 올리지만, 이 노트북은 **선생님 저장소에서 바로 읽어** 올릴 파일이 없습니다.

**문제 정의** — 자동차의 무게 · 마력 · 배기량 · 연식으로 **연비(mpg, 갤런당 마일)** 를 맞힌다.
연비는 **숫자** → **예측(회귀)** 문제 → 지도학습.

| 단계 | 이 노트북에서 | 핵심 함수 |
|---|---|---|
| ① 불러오기 · 탐색 | 행·열 · 결측치 · 요약 통계 · 그래프 | `load_dataset` `info` `describe` `isnull` |
| ② 전처리 | 결측 행 삭제, 특성 X · 정답 y 나누기 | `dropna` `df[[...]]` |
| ③ 데이터 분리 | 훈련 80% · 테스트 20% | `train_test_split` |
| ④ 모델 학습 | 선형 회귀 | `LinearRegression()` `fit` |
| ⑤ 예측 | 테스트 데이터 · 새 자동차 | `predict` |
| ⑥ 평가 | MAE · MSE · RMSE · R² · 그래프 | `mean_absolute_error` `mean_squared_error` `r2_score` |
| ⑦ 개선 | 특성 더하기 · 다른 모델과 비교 | `RandomForestRegressor` |
| ⑧ 해석 | 결과를 문장으로 | — |

In [ ]:
# (그래프에 한글이 네모로 나오면 한 번만 실행)
!pip -q install koreanize-matplotlib
import koreanize_matplotlib

---
## ① 불러오기와 탐색
`sns.load_dataset('mpg')` — seaborn 에 들어 있는 1970~80년대 자동차 398대 데이터. 올릴 파일이 없습니다.

| 열 | 뜻 |
|---|---|
| mpg | **연비(정답)** — 클수록 기름을 덜 씀 |
| cylinders · displacement | 실린더 수 · 배기량 |
| horsepower | 마력 |
| weight | 무게(파운드) |
| acceleration | 가속 시간(0→60마일, 초) |
| model_year | 연식(70 = 1970년) |
| origin · name | 생산지 · 차 이름 |

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

df = sns.load_dataset('mpg')
print(df.shape)       # (행 개수, 열 개수)
df.head()

💡 열마다 빈칸이 몇 개인지 → `isnull().sum()`

In [ ]:
print(df.____().sum())
df.describe()       # 숫자 열의 개수 · 평균 · 표준편차 · 최솟값 · 사분위수 · 최댓값

**읽기** — horsepower 에 빈칸 6개. describe 의 `mean`(평균)과 `50%`(중앙값)가 비슷하면 한쪽으로 크게 치우치지 않았다는 뜻입니다.

### 그래프로 관계 보기
연비와 무게 · 마력의 관계를 산점도로 봅니다. **오른쪽 아래로 내려가면 음의 관계**(무거울수록 연비가 낮다).

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
for a, col in zip(ax, ['weight', 'horsepower', 'model_year']):
    a.scatter(df[col], df['mpg'], s=10, alpha=0.6)
    a.set_xlabel(col); a.set_ylabel('mpg')
plt.show()

sns.heatmap(df.corr(numeric_only=True), annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1)
plt.show()

**생각해 보기 ①** 연비와 상관이 가장 큰(절댓값) 속성은? 그 부호의 뜻은?

> 적어 보기:

---
## ② 전처리 — 결측 행 삭제, X 와 y 나누기
- **X(특성 · 독립변수 · 입력)** — 힌트로 쓸 열. 두 겹 대괄호 `df[['a', 'b']]` = 열 여러 개를 표로
- **y(정답 · 종속변수 · 레이블)** — 맞힐 열. 한 겹 `df['mpg']`
💡 처음엔 특성 하나(무게)로 시작합니다 — 그래야 직선을 그림으로 볼 수 있어요.

In [ ]:
df = df.dropna()
X = df[['weight']]
y = df[____]
print(X.shape, y.shape)

---
## ③ 데이터 분리 — 훈련 데이터와 테스트 데이터 (교과서 101~102쪽)
> 🤖 «모아 둔 데이터를 전부 학습에 쓰면 평가할 데이터가 없어! **일부를 미리 떼어 두자.**» — 훈련 데이터 = 공부하는 **학습지**, 테스트 데이터 = **시험지**

`train_test_split(X, y, test_size=0.2, random_state=0)`
- `test_size=0.2` → 20%를 테스트로
- `random_state=0` → 무작위로 섞되 **누가 돌려도 같게**(재현성)
- 돌려주는 것 **네 개** → 순서 주의: `X_train, X_test, y_train, y_test`

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=____, random_state=0)
print('훈련 :', X_train.shape, ' 테스트 :', X_test.shape)

---
## ④ 모델 학습 — `fit()`
> 🤖 «`fit()` 은 단순해 보여도 그 안에서 중요한 연산이 많이 일어나! 엉성하게 틀만 있던 모델이 데이터를 학습하며 똑똑해져.»

선형 회귀는 **연비 = a × 무게 + b** 에서 오차(MSE)가 가장 작아지는 **a(기울기) · b(절편)** 를 찾습니다. 이 a, b 가 학습으로 정해지는 **파라미터**입니다.

In [ ]:
from sklearn.linear_model import LinearRegression
model = LinearRegression()
model.____(X_train, y_train)
print('기울기 a :', model.coef_[0])
print('절편   b :', model.intercept_)

**해석** — 기울기가 약 −0.0077 이면 «무게가 **1000파운드** 늘 때 연비가 약 **7.7 mpg 줄어든다**»는 뜻입니다.

### (애니메이션) 경사하강법으로 직선이 맞춰지는 모습
sklearn 은 한 번에 답을 계산하지만, 원리는 «오차가 줄어드는 쪽으로 a, b 를 조금씩 옮기기»(교과서 95쪽 경사하강법)입니다. 아래 셀을 실행하면 직선이 점점 맞아 가는 **애니메이션**이 나옵니다(▶ 누르기).

In [ ]:
import numpy as np
from matplotlib import animation
from IPython.display import HTML

x = (X_train['weight'].values - 3000) / 1000      # 계산이 쉽도록 단위 조정(1000파운드)
t = y_train.values
a, b, lr = 0.0, 0.0, 0.1                           # 처음 직선 · 학습률
hist = []
for step in range(60):
    pred = a * x + b
    err = pred - t
    hist.append((a, b, (err ** 2).mean()))
    a -= lr * 2 * (err * x).mean()                 # MSE 를 a 로 미분한 방향의 반대로
    b -= lr * 2 * err.mean()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
ax1.scatter(x, t, s=8, alpha=0.5); line, = ax1.plot([], [], 'r-', lw=2)
ax1.set_xlabel('weight (1000 lb, centered)'); ax1.set_ylabel('mpg')
ax2.set_xlim(0, 60); ax2.set_ylim(0, max(h[2] for h in hist) * 1.05); ax2.set_xlabel('step'); ax2.set_ylabel('MSE')
curve, = ax2.plot([], [], 'b-')
xs = np.linspace(x.min(), x.max(), 2)
def draw(i):
    a, b, m = hist[i]
    line.set_data(xs, a * xs + b)
    curve.set_data(range(i + 1), [h[2] for h in hist[:i + 1]])
    ax1.set_title(f'step {i}  a={a:.2f}  b={b:.2f}')
    return line, curve
anim = animation.FuncAnimation(fig, draw, frames=len(hist), interval=120)
plt.close()
HTML(anim.to_jshtml())

**보이는 것** — 오른쪽 MSE 곡선이 빠르게 내려가다 평평해집니다. 학습률 `lr` 을 0.9 로 바꾸면? 0.005 로 바꾸면? (교과서: 너무 작으면 오래 걸리고, 너무 크면 학습에 실패할 수 있다)

---
## ⑤ 예측 — `predict()`
💡 테스트 데이터의 특성 → `model.predict(X_test)`

In [ ]:
pred = model.____(X_test)
비교 = pd.DataFrame({'실제 mpg': y_test.values[:8], '예측 mpg': pred[:8].round(1)})
비교['잔차(실제-예측)'] = (비교['실제 mpg'] - 비교['예측 mpg']).round(1)
비교

**새 자동차 예측** — 입력은 학습 때와 **같은 열 이름 · 같은 모양(2차원 표)** 이어야 합니다.

In [ ]:
새차 = pd.DataFrame({'weight': [2200, 3500, 4500]})
print(model.predict(새차).round(1))

---
## ⑥ 평가 — 숫자 맞히기 모델의 성능 지표 (교과서 106~107쪽)
| 지표 | 뜻 | 좋은 방향 | 함수 |
|---|---|---|---|
| **MAE** | 평균적으로 몇 mpg 빗나갔나(절댓값 평균) | 0에 가깝게 | `mean_absolute_error(실제, 예측)` |
| **MSE** | 오차 제곱의 평균 — 큰 실수에 벌을 더 줌 | 0에 가깝게 | `mean_squared_error(실제, 예측)` |
| **RMSE** | MSE 의 제곱근 — 단위가 mpg 로 돌아옴 | 0에 가깝게 | `np.sqrt(MSE)` |
| **R²** | 연비가 달라지는 정도를 모델이 몇 % 설명하나 | 1에 가깝게 | `r2_score(실제, 예측)` 또는 `model.score(X_test, y_test)` |

💡 함수 안 순서는 늘 **(실제값, 예측값)**

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
mae = mean_absolute_error(y_test, pred)
mse = mean_squared_error(____, pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, pred)
print(f'MAE  {mae:.2f} mpg')
print(f'MSE  {mse:.2f}')
print(f'RMSE {rmse:.2f} mpg   (연비 평균 {y_test.mean():.1f} 의 {rmse / y_test.mean() * 100:.0f}%)')
print(f'R²   {r2:.3f}')
print('훈련 R²', round(model.score(X_train, y_train), 3), '/ 테스트 R²', round(model.score(X_test, y_test), 3))

### 그래프로 평가 — ① 실제 vs 예측  ② 잔차
- ① 점들이 **대각선**에 붙을수록 잘 맞힘
- ② 잔차(실제−예측)가 0 선 위아래로 **고르게** 흩어져야 좋음. 한쪽으로 **휘어진 모양**이면 직선이 놓친 패턴이 있다는 뜻

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].scatter(y_test, pred, s=12); ax[0].plot([8, 46], [8, 46], 'r--')
ax[0].set_xlabel('actual mpg'); ax[0].set_ylabel('predicted mpg'); ax[0].set_title('actual vs predicted')
ax[1].scatter(pred, y_test - pred, s=12); ax[1].axhline(0, color='r', ls='--')
ax[1].set_xlabel('predicted mpg'); ax[1].set_ylabel('residual'); ax[1].set_title('residuals')
plt.show()

**생각해 보기 ②** 잔차 그래프가 «U자»로 휘어 있나요? 그렇다면 무게와 연비의 관계가 직선이 아니라 **곡선**이라는 뜻입니다.

---
## ⑦ 개선 — 특성 더하기 · 다른 모델과 비교
같은 테스트 데이터 · 같은 지표로 비교해야 공정합니다(교과서 «모델 수정»).

In [ ]:
from sklearn.ensemble import RandomForestRegressor
특성들 = ['weight', 'horsepower', 'model_year', 'displacement', 'cylinders']
X2 = df[특성들]
X2_train, X2_test, y2_train, y2_test = train_test_split(X2, y, test_size=0.2, random_state=0)

결과 = {}
for 이름, m, Xtr, Xte in [('선형 회귀(무게 하나)', LinearRegression(), X_train, X_test),
                         ('선형 회귀(특성 5개)', LinearRegression(), X2_train, X2_test),
                         ('랜덤 포레스트(특성 5개)', RandomForestRegressor(300, random_state=0), X2_train, X2_test)]:
    m.fit(Xtr, y_train if Xtr is X_train else y2_train)
    p = m.predict(Xte)
    결과[이름] = {'RMSE': np.sqrt(mean_squared_error(y_test if Xte is X_test else y2_test, p)),
                  'R²': r2_score(y_test if Xte is X_test else y2_test, p)}
pd.DataFrame(결과).T.round(3)

In [ ]:
# 다중 선형 회귀의 계수 — «다른 특성이 같을 때» 그 특성이 1 늘면 연비가 얼마나 변하나
m5 = LinearRegression().fit(X2_train, y2_train)
pd.Series(m5.coef_, index=특성들).round(4)

**주의** — 계수의 크기는 **단위**에 따라 달라집니다(무게는 1파운드, 연식은 1년). 크기끼리 직접 비교하려면 표준화가 필요해요.

---
## ⑧ 결과 해석 — 문장으로 쓰기
> 우리 모델(______)은 테스트 자동차 ___대에서 **RMSE ___ mpg** 로, 평균적으로 약 ___ mpg 빗나간다. **R² ___** 은 연비가 달라지는 정도의 ___% 를 설명한다는 뜻이다.
> 무게 하나만 쓸 때보다 연식 · 마력을 더하니 RMSE 가 ___ 줄었다. 다만 1970~80년대 미국 · 유럽 · 일본 자동차 데이터라서 **요즘 자동차(하이브리드 · 전기차)에는 그대로 쓸 수 없다.**

---
### 정답 예시 (막혔을 때만)
- ① `isnull` · ② `'mpg'` · ③ `0.2` · ④ `fit` · ⑤ `predict` · ⑥ `y_test`
- 무게 하나: R² 약 0.7 · 특성 5개 선형: 약 0.8 · 랜덤 포레스트: 약 0.85~0.9 (나누기에 따라 조금 다름)